In [ ]:
# Import Variables from .env file
from dotenv import load_dotenv          
load_dotenv() 

True

# The Eval Loop


##### The Eval Loop
Build -> Measure -> Improve -> Repeat

#### Evaluation Types
1. Human evaluation -- Use during initial development and for periodic "ground truth" checks. Essential for high-stakes domains (medical, legal, financial).
2. Automated/deterministic -- Use as a first line of defense for every output. Check JSON schema validity, required field presence, forbidden content patterns, response length bounds.
4. Model-graded -- Use as your primary scaling mechanism. Run after every prompt change against your full eval set. Supplement with periodic human review to keep the judge calibrated.

##### Best Practices
1. Use a Different Model Instance. The generating instance retains its own reasoning context, making it biased toward its own output. Always use a separate instance (or a different model entirely) for evaluation. 
2. Include Clear Rubric Definitions. Don't just say "rate accuracy 1-5." Define what each score level means with concrete examples. What does a "3" look like? How is it different from a "4"? Without this, the judge's scores will drift over time.
3. Calibrate Against Human Judgments. Run the judge on a sample set where you have human scores. Compare the results. If the judge consistently scores 1 point higher than humans on "completeness," adjust the rubric. Track this calibration over time.
4. Track Inter-Rater Reliability. Run the same evaluation multiple times. If the judge gives the same response a 3 on one run and a 5 on another, your rubric is too vague. Consistency (low variance across runs) is as important as accuracy (matching human scores).


# 2. LLM-as-Judge
# The Judge Prompt Anatomy
You are evaluating an AI assistant's response.

Criteria:
- Accuracy (1-5): Are all facts correct?
  1 = Contains factual errors
  3 = Mostly correct with minor imprecisions
  5 = All facts verified and precise

- Completeness (1-5): Does it address all parts of the question?
  1 = Misses major aspects of the question
  3 = Addresses the main point but misses secondary aspects
  5 = Thoroughly addresses every part

- Clarity (1-5): Is it well-organized and easy to understand?
  1 = Confusing, disorganized, or overly technical
  3 = Understandable but could be better structured
  5 = Crystal clear with logical flow

- Actionability (1-5): Can the user act on this information?
  1 = Vague advice with no concrete next steps
  3 = Some actionable items but missing specifics
  5 = Clear, specific steps the user can take immediately

Response to evaluate:
[response]

Provide scores and brief justification for each criterion.
Format as JSON:
{
  "accuracy": { "score": N, "justification": "..." },
  "completeness": { "score": N, "justification": "..." },
  "clarity": { "score": N, "justification": "..." },
  "actionability": { "score": N, "justification": "..." }
}

##### 3. Building Evaluation Sets
A golden dataset should contain 20-50 representative examples with expected outputs. 
1. Easy cases (40%) -- Common, straightforward inputs that your system should handle flawlessly. These establish your baseline accuracy.
2. Hard cases (30%) -- Complex inputs that require multi-step reasoning, disambiguation, or domain expertise. These test your system's ceiling.
3. Edge cases and adversarial examples (30%) -- Tricky inputs that commonly fool AI: ambiguous phrasing, contradictory context, requests that look benign but require refusal, inputs with subtle errors.

##### 4. Context Management
###### Progressive Summarization Risks
####### Strategies for Context Management
1. Selective Injection: Only include context that is directly relevant to the current task.
2. Structured Context Blocks: For critical information that must persist across a conversation, create a dedicated "case facts" section at the top of every prompt. Format it as a clearly delimited block with labeled fields. This prevents important facts from getting lost as the conversation grow

<case_facts>
Customer: Jane Smith (ID: C-4829)
Account tier: Enterprise
Issue: Invoice #INV-2024-0312 overcharged by $1,247.50
Resolution authority: Up to $2,000 without escalation
Previous contacts: 2 (both unresolved)
</case_facts>

3. Tool Output Trimming: 
4. The /compact Command: In Claude Code, the /compact command compresses your conversation context. Use it when you notice the conversation getting long, especially before starting a new phase of work. It summarizes earlier exchanges while preserving key decisions and facts.
5. Subagent Delegation: For research-heavy tasks, delegate to a subagent with its own context window. The subagent reads 50 files, synthesizes findings, and returns a concise 300-token summary. Your main conversation never sees the 50 files, preserving context for the actual work.


##### 5. Escalation and Confidence
###### When to escalate
1. Customer explicitly asks for a human 
2. Policy gaps -- When Claude encounters a situation that the system prompt and available documentation do not address
3. Ambiguous situations with no clear right answer
4. High-stakes decisions requiring human accountability 

##### Confidence Calibration. Instead of relying on Claude's self-assessment, build confidence into your system design:
a) Use explicit criteria -- Define specific conditions under which escalation should happen, not vague "if you're unsure" instructions.
b) Provide few-shot examples -- Show Claude examples of situations that should and should not be escalated. Concrete examples are more reliable than abstract rules.
c) Source-based confidence -- An answer backed by a specific document citation is more reliable than one from general knowledge. Build this distinction into your system.
###### Handoff Patterns

<escalation_summary>
Reason: Policy gap -- customer requesting exception not covered in guidelines
Customer: Jane Smith (Enterprise tier, 3-year relationship)
Request: Waive annual renewal fee ($4,800) due to service outage
Relevant context:
  - 47-hour outage affected this customer's production environment
  - SLA guarantees 99.9% uptime (this month: 93.4%)
  - No existing policy for fee waiver vs. service credit
Attempted resolution: Offered standard 10% service credit ($480)
Customer response: Rejected, requesting full fee waiver
My assessment: Customer request seems reasonable given SLA breach severity
</escalation_summary>

###### When escalating, provide the human reviewer with a structured summary:

###### Python Agent Implementation
This example uses a structured tool-calling pattern. When the agent determines a "Policy Gap," it invokes the escalate_to_human tool.


##### 6. Error Propagation in Multi-Agent Systems
 Each agent should be able to report: what happened, what was attempted, and what partial results are available.

 {
  "status": "partial_failure",
  "completed_steps": ["extract", "validate"],
  "failed_step": "enrich",
  "error": {
    "type": "ServiceTimeout",
    "message": "Enrichment API timed out after 30s",
    "attempted": "3 retries with exponential backoff"
  },
  "partial_results": {
    "extracted_data": { /* ... available data ... */ },
    "validation_status": "passed"
  },
  "skipped_steps": ["store"],
  "suggested_action": "Retry enrichment or proceed with unenriched data"
}

###### Recovery Strategies
1. Retry with backoff -- For transient errors (timeouts, rate limits, temporary service outages).
2. Fallback to a simpler approach -- If the enrichment API fails, use the data you already have rather than failing entirely. A response with unenriched data is better than no response.
3. Continue with partial results -- If 4 of 5 data sources are available, produce a result with a disclaimer about missing information. Let the consumer decide if the partial result is sufficient.
4. Escalate to human -- For unrecoverable errors that affect business-critical decisions. Include the full error context so the human reviewer can diagnose and resolve.



In [ ]:
# Error Propagation in Multi-Agent Systems
# This example demonstrates a function that processes a lead. It uses a try-except block to capture a failure at a specific step and returns the structured JSON response you provided.

import json

def process_lead_pipeline(lead_data):
    """
    Simulates a multi-step pipeline. 
    Uses a 'State Object' to track progress for graceful error recovery.
    """
    # Initialize the response structure
    # This acts as our 'flight recorder' for the process
    state = {
        "status": "success",
        "completed_steps": [],
        "failed_step": None,
        "error": None,
        "partial_results": {},  # Stores data gathered before any crash
        "skipped_steps": [],
        "suggested_action": None
    }

    try:
        # --- STEP 1: EXTRACT ---
        # Goal: Get raw data. Even if later steps fail, we want to keep this.
        state["partial_results"]["extracted_data"] = {"email": lead_data.get("email")}
        state["completed_steps"].append("extract")

        # --- STEP 2: VALIDATE ---
        # Goal: Ensure data quality. 
        state["partial_results"]["validation_status"] = "passed"
        state["completed_steps"].append("validate")

        # --- STEP 3: ENRICH (The Fragile Step) ---
        # Simulating an external API call that might be unreliable or slow.
        # In a real scenario, this might be a request to a service like Clearbit or ZoomInfo.
        raise TimeoutError("Enrichment API timed out after 30s")
        
        # If the code reached here, we would mark enrichment as done
        state["completed_steps"].append("enrich")
        
        # --- STEP 4: STORE ---
        # Final step: Saving to a database.
        state["completed_steps"].append("store")

    except TimeoutError as e:
        # If a timeout occurs, we 'catch' it to prevent the whole app from crashing.
        state["status"] = "partial_failure"
        state["failed_step"] = "enrich"
        
        # We provide 'Metadata' about the error so the system knows IF it should retry.
        state["error"] = {
            "type": "ServiceTimeout",
            "message": str(e),
            "attempted": "3 retries with exponential backoff"
        }
        
        # Explicitly list what didn't happen so the UI/Orchestrator can flag it.
        state["skipped_steps"] = ["store"]
        
        # Provide a 'Human-Readable' or 'Agent-Readable' suggestion for the next move.
        state["suggested_action"] = "Retry enrichment or proceed with unenriched data"

    except Exception as e:
        # Catch-all for unexpected logic errors
        state["status"] = "total_failure"
        state["error"] = {"type": "UnexpectedError", "message": str(e)}

    return state

# --- Execution Logic ---
raw_input = {"email": "jane@enterprise.com"}
result = process_lead_pipeline(raw_input)

# Check the status to decide how to handle the output
if result["status"] == "partial_failure":
    # Log the failure for developer monitoring
    print(f"DEBUG: Pipeline stalled at {result['failed_step']}.")
    
    # We can still use the data we DID get!
    safe_data = result['partial_results']['extracted_data']
    print(f"ACTION: Proceeding with base data for: {safe_data['email']}")

# Final JSON output for the API response
print(json.dumps(result, indent=2))

* Design principle: Every agent should be able to run independently with degraded input and produce a partial result. If your pipeline requires every step to succeed for any output to be produced, your architecture is fragile.

In [ ]:
# Claim-Source Mappings
# The gold standard for provenance is a claim-source mapping: each assertion in the output is linked to the specific evidence that supports it.
# This format is known as Grounded Response with Citations. It is used when accuracy and auditability are non-negotiable.
# By separating the claims (the output text) from the evidence (the sources), you allow a human or a supervisor agent to verify facts without hunting through raw data files.
# Legal or Financial Analysis:When an agent is quoting policy or account history to a frustrated customer; citations provide "receipts" that build trust.
# High-Stakes Support: When an agent is quoting policy or account history to a frustrated customer; citations provide "receipts" that build trust.
# Multi-Document Synthesis: When the agent is pulling info from three different places (e.g., a PDF, a Database, and an API) and needs to prove where each specific fact came from.
# Reducing Hallucinations: Using this structure forces the AI to "anchor" its response to specific data points, making it much harder for the model to make up dates or numbers.

# Python Code Example: The Grounded Response Pattern
# This example demonstrates how to build a function that synthesizes a response and maps specific "claims" to their original "data sources."

import json

def generate_grounded_renewal_report(account_id):
    """
    Simulates fetching data from multiple sources and generating 
     a response where every claim is tied to a source index.
    """
    
    # 1. Simulate data retrieval from diverse sources
    # In a real app, these would be API calls or Database queries
    data_sources = {
        "contract": {"expiry": "March 15, 2025", "ref": "Contract document, Section 4.2"},
        "crm": {"tenure": "3 years", "created": "2022-03-15"},
        "billing": {"renewal_price": "$48,000/year", "api_timestamp": "2025-01-10"}
    }

    # 2. Construct the individual claims
    # Each claim is a sentence tied to a specific numbered source
    claims = [
        f"The customer's contract expires {data_sources['contract']['expiry']} [1].",
        f"Their account has been active for {data_sources['crm']['tenure']} [2].",
        f"Based on current pricing, renewal would be {data_sources['billing']['renewal_price']} [3]."
    ]

    # 3. Compile the Sources list
    # This acts as the 'Bibliography' for the response
    sources = {
        "[1]": f"{data_sources['contract']['ref']}, paragraph 1",
        "[2]": f"CRM record, account_created_at: {data_sources['crm']['created']}",
        "[3]": f"Pricing API response, enterprise_annual tier, retrieved {data_sources['billing']['api_timestamp']}"
    }

    # 4. Final structured output
    # This format is perfect for front-end UI components that show tooltips for citations
    report = {
        "output_text": " ".join(claims),
        "sources": sources
    }

    return report

# --- Execution ---

result = generate_grounded_renewal_report("ACC-9921")

# Displaying the result
print("--- AGENT RESPONSE ---")
print(result["output_text"])

print("\n--- SOURCES / VERIFICATION ---")
for key, value in result["sources"].items():
    print(f"{key}: {value}")

# Example of how you'd use this in a JSON-based API
# print(json.dumps(result, indent=2))